### Electric Vehicle Routing Problem with Non-Linear Charging and Capacitated Charging Stations
- Objetivo: Minimizar o tempo total para atender todos os clientes, incluindo tempo de deslocalemento, tempo de realização dos serviços, tempo de carregamento nas estações e tempo de espera para utilizar as estações de recarga
- Bibliografia base: Motoya 2017 e Groger 2022

##### Bibliotecas

In [2]:
import os
import xml.etree.ElementTree as ET
from shutil import copytree
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

#### INSTÂNCIAS
- Importação das instâncias
- Ajuste para transformar instâncias de EVRP-NL em EVRP-NL-C

In [3]:
input_instancias = "C:\\Users\laura\\OneDrive\\Área de Trabalho\\IC - EVRP-NL-C\EVRP-NL\\montoya-et-al-2017"
output_instancias = "C:\\Users\laura\OneDrive\\Área de Trabalho\\IC - EVRP-NL-C\\instancias-evrp-nl-c"

# Função para alterar o número de carregadores nas estações de carregamento (CSs)
def adj_instancia(xml_content, num_chargers):
    tree = ET.ElementTree(ET.fromstring(xml_content))
    root = tree.getroot()
    # Estações de carregamento são nós com type=2
    for node in root.find("network").find("nodes").findall("node"):
        if node.attrib["type"] == "2":
            chargers_tag = ET.SubElement(node.find("custom"), "chargers")
            chargers_tag.text = str(num_chargers)
    return ET.tostring(root, encoding='unicode')

# Loop para processar cada arquivo na pasta de inputs, ajustar para o caso da EVRP-NL-C e salvar na pasta de output
for filename in os.listdir(input_instancias):
    if filename.endswith(".xml"):
        file_path = os.path.join(input_instancias, filename)
        with open(file_path, 'r', encoding='utf-8') as file:
            xml_content = file.read()

        # Modifica criando a versão com 1 carregador
        versao_1 = adj_instancia(xml_content, 1)
        nome_versao_1 = filename.replace(".xml", "-C1.xml")
        output_versao_1 = os.path.join(output_instancias, nome_versao_1)

        # Modifica criando a versão com 2 carregadores
        versao_2 = adj_instancia(xml_content, 2)
        nome_versao_2 = filename.replace(".xml", "-C2.xml")
        output_versao_2 = os.path.join(output_instancias, nome_versao_2)

        # Salva os arquivos modificados na nova pasta
        with open(output_versao_1, 'w', encoding='utf-8') as new_file_1:
            new_file_1.write(versao_1)

        with open(output_versao_2, 'w', encoding='utf-8') as new_file_2:
            new_file_2.write(versao_2)

print(f"Processo concluído!")

Processo concluído!


#### MODELAGEM INSTÂNCIAS
- Conectando os nós das instâncias
  - Considerando de base a regra da desigualdade do triângulo
  - Usando distância euclidiana
  - Nó de depósito conectado aos 5 nós mais próximos
  - Outros nós conectados ao 3 nós mais próximos (obs.: todos são conectados aos 3 nós mais próximos então possivelmente tem nós com mais de 5 conexões, porque não tô considerando como número máximo de conexões e sim como mínimo)

In [4]:
# Função para calcular a distância euclidiana
def dist_euclidiana(node1, node2):
    return np.sqrt((node1['cx'] - node2['cx'])**2 + (node1['cy'] - node2['cy'])**2)

# Função para extrair as informações principais das instâncias (id, cx e cy) - servirá de input para contruir o grafo
def extrair_nos_xml(arquivo_xml):
    tree = ET.parse(arquivo_xml)
    root = tree.getroot()
    nodes = {}

    for node in root.find('.//nodes'):
        node_id = node.get('id')
        node_type = node.get('type')
        cx = float(node.find('cx').text)
        cy = float(node.find('cy').text)

        nodes[node_id] = {
            'cx': cx,
            'cy': cy,
            'type': int(node_type)
        }

    return nodes

# Função para criar um grafo com base nos nós
# Essa função recebe como parâmetro uma lista com infos do tipo: "id": {"cx": 9.33, "cy": 59.9, "type": 0}
def grafo(nodes):
    grafo = nx.Graph()
    # node_type (0,1 ou 2, indicando se é cliente/depósito/CS) e node_coord (coordenadas do nó cx e cy)
    for node_type, node_coord in nodes.items():
        grafo.add_node(node_type, pos=(node_coord['cx'], node_coord['cy']), type=node_coord['type'])
    node_ids = list(nodes.keys())

    # Conecta o nó 0 aos 5 nós mais próximos
    distances_to_node_0 = []
    for node_id in node_ids:
        if node_id != "0": 
            dist = dist_euclidiana(nodes["0"], nodes[node_id])
            distances_to_node_0.append((dist, node_id))

    distances_to_node_0.sort(key=lambda x: x[0])  
    closest_to_zero = distances_to_node_0[:5] 

    # Adiciona as arestas para os 5 nós mais próximos do nó 0
    for dist, closest_node in closest_to_zero:
        grafo.add_edge("0", closest_node, weight=dist)

    # Para os outros nós (exceto o nó 0), encontra os três nós mais próximos e adiciona arestas
    for node_id in node_ids:
        if node_id != "0":
            distances = []
            for other_node_id in node_ids:
                if other_node_id != node_id and other_node_id != "0": 
                    dist = dist_euclidiana(nodes[node_id], nodes[other_node_id])
                    distances.append((dist, other_node_id))
            distances.sort(key=lambda x: x[0]) 
            closest_nodes = distances[:3] 

            # Adiciona arestas para os dois nós mais próximos
            for dist, closest_node in closest_nodes:
                grafo.add_edge(node_id, closest_node, weight=dist)

    return grafo

# Função para desenhar o grafo
def draw_graph(G):
    pos = nx.get_node_attributes(G, 'pos')
    node_colors = ['red' if G.nodes[node]['type'] == 0 else 'blue' if G.nodes[node]['type'] == 1 else 'green' for node in G.nodes]
    
    # Desenha os nós
    nx.draw(G, pos, node_size=500, node_color=node_colors, with_labels=True, font_weight='bold')
    
    # Desenha as arestas com os pesos
    labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels={k: f'{v:.2f}' for k, v in labels.items()})
    
    plt.show()

#### MODELAGEM PROBLEMA
Usando como base a modelagem do artigo de Froger 2022
- Modelando as variáveis e seus domínios
    - Variáveis de Decisão:
        - t(p) é o tempo para percorrer a rota 𝑝;
        - Δpl é a duração da operação de recarga na posição 𝑙 da rota 𝑝;
        - ∇pl é o tempo de espera para iniciar a recarga na posição 𝑙 da rota 𝑝; e
        - 𝑔𝑖 é o tempo de serviço no cliente 𝑖.
- Função Objetivo

In [5]:
# Função para extrair o conjunto de clientes (I)
def extrair_clientes(arquivo_xml):
    clientes = []
    for node in root.find('network').find('nodes').findall('node'):
        node_type = int(node.get('type'))
        if node_type == 1:  # Clientes são type=1
            clientes.append(int(node.get('id')))
    return clientes

# Função para extrair estações de recarga (F)
def extrair_estacoes_recarga(arquivo_xml):
    estacoes_recarga = {}
    for node in root.find('network').find('nodes').findall('node'):
        node_type = int(node.get('type'))
        if node_type == 2:  # Estações de recarga são type=2
            chargers = int(node.find('custom').find('chargers').text)
            estacoes_recarga[int(node.get('id'))] = chargers
    return estacoes_recarga

# Função para extrair o tempo de serviço dos clientes e calcular o tempo total (gi e ∑gi)
def extrair_tempo_servico(arquivo_xml):
    tempo_servico = {}
    tempo_total_servico = 0
    for request in root.find('requests').findall('request'):
        node_id = int(request.get('node'))
        service_time = float(request.find('service_time').text)
        tempo_servico[node_id] = service_time
        tempo_total_servico += service_time
    return tempo_servico, tempo_total_servico


In [6]:
# Função principal para gerenciar a extração de dados
def parse_instance(arquivo_xml):
    tree = ET.parse(arquivo_xml)
    root = tree.getroot()
    clientes = extrair_clientes(root)
    estacoes_recarga = extrair_estacoes_recarga(root)
    tempo_servico, tempo_total_servico = extrair_tempo_servico(root)

    return clientes, estacoes_recarga, tempo_servico, tempo_total_servico

In [7]:
# Definindo variáveis fixas do problema
# Capacitade da bateria dos veículos, li na internet que era entre 24 e 90 kWh
bateria_q = 57
# Consumo médio por Km rodado em kWh (carro elétrico tem consumo médio de 20 kWh para percorrer 100 km)
energia_consumo_e = 0.2
# Vou considerar velocidade média de 60km/h
velocidade_media = 60
# Definição aleatória em horas
tempo_max = 8
# Breakpoints - 50% e 80%
breakpoints = [0.5, 0.8]

In [8]:
def carregamento_base_carga(soc_desejado, soc_inicial, bateria_q):
    """
    - soc_desejado: nível kWh que queremos atingir nesse carregamento
    - soc_inicial: estado de carga (SoC) inicial do EV ao chegar em j (em %)
    - bateria_q: capacidade máxima da bateria do EV (em kWh)
    """
    # Ajustando o SoC inicial
    soc_inicial_adj = (soc_inicial / bateria_q)  # Convertendo de kWh para proporção
    soc_desejado_adj = (soc_desejado / bateria_q)  # Convertendo kWh desejado para proporção

    # Função interna para calcular a taxa de recarga com base no SoC
    def phi_j(soc_inicial_adj):
        # Ajuste da taxa de carga dependendo do SoC
        if soc_inicial_adj <= 0.5:  # 50% de SoC
            return 50 / 100  # Carga de até 50% (50% por hora)
        elif soc_inicial_adj <= 0.8:  # 50% a 80% de SoC
            return 30 / 100  # Carga de 50% a 80% (30% por hora)
        else:
            return 10 / 100  # Acima de 80% (10% por hora)

    # Inicializando variáveis
    tempo_total = 0  # Tempo total em horas
    soc_atual = soc_inicial_adj

    # Loop até atingir o SoC desejado
    while soc_atual < soc_desejado_adj:
        # Calcula a taxa de recarga atual
        taxa_recarregamento = phi_j(soc_atual)

        # Calcula quanto tempo leva para alcançar a próxima carga de 5% da bateria
        carga_para_aumentar = 0.02  # Aumentar em 5% do total da bateria
        tempo_necessario = carga_para_aumentar / taxa_recarregamento  # Tempo em horas para carregar 5%

        # Atualiza o SoC atual
        soc_atual += carga_para_aumentar
        
        # Atualiza o tempo total
        tempo_total += tempo_necessario
        
        # Exibe o estado atual
        print(f"O SoC atual é: {soc_atual * 100:.2f}%, Tempo total: {tempo_total:.2f} h, Taxa de recarregamento: {taxa_recarregamento * 100:.2f}%")

    return tempo_total  # Retorna o tempo total necessário para atingir o SoC desejado

In [9]:
def carregamento_base_tempo(delta, soc_inicial, bateria_q):
    """
    - delta: tempo gasto carregando em horas
    - soc_inicial: estado de carga inicial do EV ao chegar (em kWh)
    - bateria_q: capacidade máxima da bateria do EV (em kWh)
    """
    # Ajustando o SoC inicial
    soc_inicial_adj = soc_inicial / bateria_q  # Convertendo de kWh para proporção

    # Função para calcular a taxa de recarga com base no SoC atual
    def phi_j(soc_atual_adj):
        if soc_atual_adj <= 0.5:      # Até 50% do SoC
            return 0.50               # Taxa de recarga de 50% por hora
        elif soc_atual_adj <= 0.8:    # De 50% a 80% do SoC
            return 0.30               # Taxa de recarga de 30% por hora
        else:                         # Acima de 80% do SoC
            return 0.10               # Taxa de recarga de 10% por hora

    # Inicializando variáveis
    soc_atual = soc_inicial_adj  # Estado de carga atual em proporção
    tempo_restante = delta       # Tempo restante para carregamento

    # Loop para incrementar o SoC até esgotar o tempo de carregamento
    while tempo_restante > 0 and soc_atual < 1.0:
        taxa_recarregamento = phi_j(soc_atual)      # Taxa de recarga para o SoC atual
        tempo_para_aumentar_5 = 0.05 / taxa_recarregamento  # Tempo necessário para aumentar 5% do SoC
        
        # Verifica se há tempo suficiente para o próximo aumento de 5%
        if tempo_para_aumentar_5 <= tempo_restante:
            soc_atual += 0.05                      # Incrementa o SoC em 5%
            tempo_restante -= tempo_para_aumentar_5  # Atualiza o tempo restante
        else:
            # Incremento final proporcional ao tempo restante
            soc_atual += taxa_recarregamento * tempo_restante
            tempo_restante = 0  # Tempo esgotado

        # Exibe o estado de carga e o tempo restante
        print(f"SoC atual: {soc_atual * 100:.2f}%, Tempo restante: {tempo_restante:.2f} h, Taxa de recarga: {taxa_recarregamento * 100:.2f}%")

    # Converte o SoC final para kWh
    soc_final_kwh = soc_atual * bateria_q
    return soc_atual * 100, soc_final_kwh  # Retorna o SoC final em % e em kWh

In [10]:
model = gp.Model("EVRP-NL-C")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2537843


GurobiError: License 2537843 has expired

In [ ]:
# Conjunto de Dados - Dúvida de como definir essas rotas
P = [...]  # Conjunto de rotas
n_p = {...}  # Número de estações de recarga na rota p

# Conjunto de clientes, tempo para percorrer a rota p e tempo de serviço no cliente i, respectivamente
# I, t_p, g_i = parse_instance(arquivo_xml)
# Comentado por não estar chamando nada, apenas declarado

In [ ]:
# Variáveis
x = model.addVars(P, vtype=GRB.BINARY, name="x")  # Variável binária x(p)
Delta = model.addVars(P, n_p, lb=0.0, name="Delta")  # Tempo de recarga Δpl
Nabla = model.addVars(P, n_p, lb=0.0, name="Nabla")  # Tempo de espera ∇pl
G = model.addVars(I, lb=0.0, name="g")  # Tempo de serviço Gi